In [0]:
dbutils.widgets.text("catalog","sl")
catalog = dbutils.widgets.get("catalog")

#passenger

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Ensure the gold schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")

# Source: validated passengers from silver (one row per passenger enforced there)
passengers = spark.table(f"{catalog}.silver.passengers") \
    .select(
        col("pa_passengerid").alias("passenger_id"),
        trim(col("pa_name")).alias("passenger_name"),
        trim(col("pa_address")).alias("passenger_address"),
        col("pa_nationkey").alias("nation_key"),
        trim(col("pa_phone")).alias("passenger_phone"),
        col("pa_loyaltypoints").alias("loyalty_points"),
        col("pa_travelpurpose").alias("travel_purpose"),
        col("pa_comment").alias("passenger_comment"),
    )

w = Window.partitionBy("passenger_id").orderBy(lit(1))
dim_passenger = passengers.withColumn("row_num", row_number().over(w)) \
    .filter(col("row_num") == 1) \
    .drop("row_num")

dim_passenger.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.gold.dim_passenger")




In [0]:
print(f"Rows written to {catalog}.gold.dim_passenger: {dim_passenger.count()}")

#dim_route

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window


routes = spark.table(f"{catalog}.silver.route") \
    .select(
        col("ro_routeid").alias("route_id"),
        col("ro_name").alias("route_name"),
        trim(col("ro_aircraft")).alias("aircraft"),
        trim(col("ro_airline_brand")).alias("airline_brand"),
        col("ro_routetype").alias("route_type"),
        col("ro_capacity").alias("capacity"),
        col("ro_baseprice").alias("base_price"),
        col("ro_cabinconfig").alias("cabin_config"),
        trim(col("ro_comment")).alias("route_comment"),
    )

# Safeguard: keep exactly one row per route_id
w = Window.partitionBy("route_id").orderBy(lit(1))
dim_route = routes.withColumn("row_num", row_number().over(w)) \
    .filter(col("row_num") == 1) \
    .drop("row_num")

dim_route.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.gold.dim_route")




In [0]:
print(f"Rows written to {catalog}.gold.dim_route: {dim_route.count()}")

#dim_airline

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Source: validated airlines from silver (one row per airline enforced there)
airlines = spark.table(f"{catalog}.silver.airline") \
    .select(
        col("al_airlineid").alias("airline_id"),
        trim(col("al_name")).alias("airline_name"),
        trim(col("al_address")).alias("airline_address"),
        col("al_nationkey").alias("nation_key"),
        trim(col("al_phone")).alias("airline_phone"),
        col("al_rating").alias("rating"),
        trim(col("al_comment")).alias("airline_comment"),
    )

# Safeguard: keep exactly one row per airline_id
w = Window.partitionBy("airline_id").orderBy(lit(1))
dim_airline = airlines.withColumn("row_num", row_number().over(w)) \
    .filter(col("row_num") == 1) \
    .drop("row_num")

dim_airline.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.gold.dim_airline")




In [0]:
print(f"Rows written to {catalog}.gold.dim_airline: {dim_airline.count()}")


#dim_geography

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Source: validated nations and regions from silver
nations = spark.table(f"{catalog}.silver.nation") \
    .select(
        col("n_nationkey").alias("nation_key"),
        trim(col("n_name")).alias("nation_name"),
        col("n_regionkey").alias("region_key"),
        trim(col("n_comment")).alias("nation_comment"),
    )

regions = spark.table(f"{catalog}.silver.region") \
    .select(
        col("r_regionkey").alias("region_key"),
        trim(col("r_name")).alias("region_name"),
        trim(col("r_comment")).alias("region_comment"),
    )

# One row per nation, enriched with its parent region
dim_geography = nations.join(regions, on="region_key", how="left") \
    .select(
        "nation_key",
        "nation_name",
        "nation_comment",
        "region_key",
        "region_name",
        "region_comment",
    )

# Safeguard: keep exactly one row per nation_key
w = Window.partitionBy("nation_key").orderBy(lit(1))
dim_geography = dim_geography.withColumn("row_num", row_number().over(w)) \
    .filter(col("row_num") == 1) \
    .drop("row_num")

dim_geography.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.gold.dim_geography")

print(f"Rows written to {catalog}.gold.dim_geography: {dim_geography.count()}")
display(dim_geography.limit(5))

#fact_bookings

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Source: validated flight segment booking lines from silver
# Grain: one row per booking line (booking_id + line_number)
flight_lines = spark.table(f"{catalog}.silver.flight_booking") \
    .select(
        col("fb_bookingid").alias("booking_id"),
        col("fb_linenumber").alias("line_number"),
        col("fb_routeid").alias("route_id"),
        col("fb_airlineid").alias("airline_id"),
        col("fb_seats").alias("seats"),
        col("fb_ticketprice").alias("ticket_price"),
        col("fb_discount").alias("discount"),
        col("fb_tax").alias("tax"),
        trim(col("fb_refundflag")).alias("refund_flag"),
        trim(col("fb_segmentstatus")).alias("segment_status"),
        col("fb_departuredate").alias("departure_date"),
        col("fb_commitdate").alias("commit_date"),
        col("fb_arrivaldate").alias("arrival_date"),
        trim(col("fb_cabinclass")).alias("cabin_class"),
        trim(col("fb_comment")).alias("line_comment"),
    )

# Parent booking attributes (one row per booking in silver)
bookings = spark.table(f"{catalog}.silver.booking") \
    .select(
        col("bk_bookingid").alias("booking_id"),
        col("bk_passengerid").alias("passenger_id"),
        trim(col("bk_status")).alias("booking_status"),
        col("bk_bookingdate").alias("booking_date"),
        col("bk_flightdate").alias("flight_date"),
        col("bk_bookingpriority").alias("booking_priority"),
        trim(col("bk_agent")).alias("agent"),
    )

# Enrich each segment line with its parent booking attributes
fact_booking = flight_lines.join(bookings, on="booking_id", how="left") \
    .withColumn(
        "gross_amount",
        round(col("ticket_price") * col("seats"), 2),
    ) \
    .withColumn(
        "net_amount",
        round(col("ticket_price") * (lit(1) - col("discount")) * col("seats"), 2),
    )

# Safeguard: keep exactly one row per booking line (composite key)
w = Window.partitionBy("booking_id", "line_number").orderBy(lit(1))
fact_booking = fact_booking.withColumn("row_num", row_number().over(w)) \
    .filter(col("row_num") == 1) \
    .drop("row_num")

fact_booking.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.gold.fact_booking")



In [0]:
print(f"Rows written to {catalog}.gold.fact_booking: {fact_booking.count()}")


#vw_annual_booking_kpis

In [0]:
# Annual booking KPIs: revenue, distinct bookings, active passengers, avg booking value
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.gold.vw_annual_booking_kpi AS
SELECT
    year(bk_bookingdate)                                        AS booking_year,
    COUNT(DISTINCT bk_bookingid)                                AS distinct_bookings,
    COUNT(DISTINCT bk_passengerid)                              AS active_passengers,
    SUM(bk_totalprice)                                          AS annual_revenue,
    ROUND(SUM(bk_totalprice) / COUNT(DISTINCT bk_bookingid), 2) AS avg_booking_value
FROM {catalog}.silver.booking
WHERE year(bk_bookingdate) IS NOT NULL
GROUP BY year(bk_bookingdate)
""")

display(spark.table(f"{catalog}.gold.vw_annual_booking_kpi"))